# cells

> The SolveIt-style cell GUI: a stack of typed cells (code / note / prompt / raw),
each toggleable in/out of the LLM's view. Because the model call is stateless,
context is just re-assembled from whichever cells are currently visible.

Note cells are markdown (KaTeX math, images, raw HTML). Standard Jupyter
command-mode hotkeys drive selection and editing.

In [ ]:
#| default_exp cells

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
import fasthtml.components as fh
from toolslm.shell import get_shell
from datetime import datetime
import nbformat as _nbf

## DaisyUI + app setup

CDN headers for DaisyUI + Tailwind, plus `KatexMarkdownJS()` (renders `.marked`
elements as markdown + KaTeX) and the command-mode hotkey listener.

In [ ]:
#| export
# Jupyter-style command-mode hotkeys. Fire only when no textarea/input is focused;
# each maps to an htmx POST that re-renders #notebook.
_HOTKEYS_JS = r"""
(function(){
  let lastD = 0;
  const act = (url) => htmx.ajax('POST', url, {target:'#notebook', swap:'outerHTML'});

  // Toggle '# ' comments on the selected lines of a textarea (Cmd/Ctrl+/).
  const toggleComment = (ta) => {
    const v = ta.value, s = ta.selectionStart, e = ta.selectionEnd;
    const ls = v.lastIndexOf('\n', s - 1) + 1;
    let le = v.indexOf('\n', e); if (le === -1) le = v.length;
    const lines = v.slice(ls, le).split('\n');
    const commented = lines.every(l => l.trim() === '' || l.trimStart().startsWith('#'));
    const out = commented
      ? lines.map(l => l.replace(/^(\s*)#\s?/, '$1')).join('\n')
      : lines.map(l => l.trim() === '' ? l : l.replace(/^(\s*)/, '$1# ')).join('\n');
    ta.value = v.slice(0, ls) + out + v.slice(le);
    ta.selectionStart = ls; ta.selectionEnd = ls + out.length;
  };

  document.addEventListener('keydown', (e) => {
    const a = document.activeElement;
    if (a && a.closest && a.closest('.CodeMirror')) return;  // CodeMirror handles its own keys
    const inEditor = a && (a.tagName === 'TEXTAREA' || a.tagName === 'INPUT' || a.isContentEditable);
    const mod = e.metaKey || e.ctrlKey;

    // --- editor shortcuts (fire while typing) ---
    if (mod && !e.shiftKey && e.key === '/') {
      if (a && a.tagName === 'TEXTAREA') { toggleComment(a); e.preventDefault(); }
      return;
    }
    if (mod && e.shiftKey && (e.code === 'Minus' || e.key === '-' || e.key === '_')) {
      if (a && a.id === 'compose-input') {
        htmx.ajax('POST', '/split', {target:'#app', swap:'outerHTML',
                   values:{source: a.value, pos: a.selectionStart}});
        e.preventDefault();
      }
      return;
    }

    // --- command-mode shortcuts (only when not editing) ---
    if (inEditor || e.metaKey || e.ctrlKey || e.altKey) return;
    let handled = true;
    switch (e.key) {
      case 'a': act('/insert?where=above'); break;
      case 'b': act('/insert?where=below'); break;
      case 'j': case 'ArrowDown': act('/select_delta?delta=1');  break;
      case 'k': case 'ArrowUp':   act('/select_delta?delta=-1'); break;
      case 'm': act('/settype_selected?t=note'); break;
      case 'y': act('/settype_selected?t=code'); break;
      case 'r': act('/settype_selected?t=raw');  break;
      case 's': htmx.ajax('POST', '/save_now', {swap:'none'}); break;
      case 'd': { const n = Date.now();
        if (n - lastD < 500) { lastD = 0; act('/del_selected'); }
        else { lastD = n; handled = false; }
        break; }
      default: handled = false;
    }
    if (handled) e.preventDefault();
  });
})();
"""

# Tailwind's reset flattens markdown headings/lists; restore them for `.marked` content.
_MARKED_CSS = """
.marked h1{font-size:1.6rem;font-weight:700;margin:.4em 0}
.marked h2{font-size:1.35rem;font-weight:700;margin:.4em 0}
.marked h3{font-size:1.15rem;font-weight:600;margin:.4em 0}
.marked h4{font-size:1.05rem;font-weight:600;margin:.4em 0}
.marked ul{list-style:disc;margin:.3em 0 .3em 1.5rem}
.marked ol{list-style:decimal;margin:.3em 0 .3em 1.5rem}
.marked p{margin:.4em 0}
.marked a{color:#2563eb;text-decoration:underline}
.marked code{background:rgba(127,127,127,.2);padding:.1em .3em;border-radius:.25rem;font-family:monospace}
.marked pre{background:rgba(127,127,127,.15);padding:.6em;border-radius:.4rem;overflow:auto}
.marked pre code{background:none;padding:0}
.marked blockquote{border-left:3px solid #999;padding-left:.75em;margin:.4em 0;opacity:.85}
.marked table{border-collapse:collapse}
.marked th,.marked td{border:1px solid #999;padding:.2em .5em}
.marked img{max-width:100%}
.CodeMirror{height:auto;border:1px solid #ccc;border-radius:.4rem;font-size:.85rem}
.CodeMirror-scroll{max-height:60vh}
"""

_EDIT_JS = r"""
function boopSave(id){
  var cm = window['_boopcm_'+id];
  if(cm) cm.save();
  window._boopFocusAfter = id;
  var ta = document.getElementById('ta-'+id);
  if(ta && ta.form) ta.form.requestSubmit();
}
function boopComposerSubmit(){
  var ta = document.getElementById('compose-input');
  var cm = window._boopComposerCM;
  if(cm && cm.getWrapperElement && cm.getWrapperElement().isConnected) cm.save();
  if(ta && ta.form) ta.form.requestSubmit();
}
function boopComposerSplit(cm){
  htmx.ajax('POST', '/split', {target:'#notebook', swap:'beforeend',
    values:{source: cm.getValue(), pos: cm.indexFromPos(cm.getCursor())}});
}
function boopMakeCM(ta, isComposer){
  var dark = window._boopDark !== false;
  var id = ta.getAttribute('data-cid');
  var extra = isComposer ? {
      'Shift-Enter': boopComposerSubmit, 'Ctrl-Enter': boopComposerSubmit, 'Cmd-Enter': boopComposerSubmit,
      'Ctrl-/': function(cm){ cm.toggleComment(); }, 'Cmd-/': function(cm){ cm.toggleComment(); },
      'Shift-Ctrl--': function(cm){ boopComposerSplit(cm); }, 'Shift-Cmd--': function(cm){ boopComposerSplit(cm); }
    } : {
      'Shift-Enter': function(){ boopSave(id); }, 'Ctrl-Enter': function(){ boopSave(id); }, 'Cmd-Enter': function(){ boopSave(id); },
      'Ctrl-/': function(cm){ cm.toggleComment(); }, 'Cmd-/': function(cm){ cm.toggleComment(); }
    };
  var cm = CodeMirror.fromTextArea(ta, {
    mode:'python', theme: dark?'material-darker':'default',
    lineNumbers:true, lineWrapping:false, viewportMargin:Infinity, indentUnit:4, extraKeys: extra
  });
  if(isComposer){ window._boopComposerCM = cm; cm.focus(); }
  else {
    window['_boopcm_'+id] = cm; (window._boopcms = window._boopcms || []).push(cm);
    if(String(window._boopFocusAfter) === String(id)){
      window._boopFocusAfter = null; cm.focus(); cm.setCursor(cm.lineCount(), 0);
    }
  }
  return cm;
}
function boopCopy(id, ctype){
  var cm = window['_boopcm_'+id];
  var text;
  if(ctype === 'code' && cm){ text = cm.getValue(); }
  else {
    var btn = document.getElementById('copy-'+id);
    text = btn ? (btn.getAttribute('data-src') || '') : '';
  }
  navigator.clipboard.writeText(text).catch(function(){});
}
function boopRenderAnsi(root){
  if(!window.AnsiUp) return;  // module still loading; a later retry will pick these up
  var scope = (root && root.querySelectorAll) ? root : document;
  scope.querySelectorAll('.ansi-out:not([data-ansi-done])').forEach(function(el){
    el.setAttribute('data-ansi-done', '1');
    var au = new window.AnsiUp(); au.use_classes = false;
    el.innerHTML = au.ansi_to_html(el.textContent);
  });
}
function boopInitEditors(root){
  var scope = (root && root.querySelectorAll) ? root : document;
  scope.querySelectorAll('textarea[data-cm]:not([data-cminit])').forEach(function(ta){
    ta.setAttribute('data-cminit', '1');
    var kind = ta.getAttribute('data-cm');
    if(kind === 'code'){ if(window.CodeMirror) boopMakeCM(ta, false); }
    else if(kind === 'composer'){ if(window.CodeMirror) boopMakeCM(ta, true); }
    else if(kind === 'edit'){
      ta.focus(); ta.setSelectionRange(ta.value.length, ta.value.length);
      ta.addEventListener('keydown', function(e){
        if((e.shiftKey||e.ctrlKey||e.metaKey) && e.key === 'Enter'){ e.preventDefault(); boopSave(ta.getAttribute('data-cid')); }
      });
    }
  });
  boopRenderAnsi(scope);
}
if(window.htmx) htmx.onLoad(boopInitEditors);
"""

_THEME_JS = r"""
window._boopcms = window._boopcms || [];
var _HLJS = {
  dark:  'https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/styles/atom-one-dark.min.css',
  light: 'https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/styles/atom-one-light.min.css'
};
function boopApplyTheme(dark){
  window._boopDark = dark;
  document.documentElement.setAttribute('data-theme', dark ? 'dark' : 'light');
  var hl = document.getElementById('hljs-theme');
  if(hl) hl.href = dark ? _HLJS.dark : _HLJS.light;
  window._boopcms.forEach(function(cm){ cm.setOption('theme', dark ? 'material-darker' : 'default'); });
  if(window._boopComposerCM) window._boopComposerCM.setOption('theme', dark ? 'material-darker' : 'default');
  try { localStorage.setItem('boopDark', dark ? '1' : '0'); } catch(e){}
}
function boopThemeToggle(cb){ boopApplyTheme(cb.checked); }
if (window.htmx && window.hljs) {
  htmx.onLoad(function(){ document.querySelectorAll('pre code:not([data-highlighted=\"yes\"])').forEach(function(c){ hljs.highlightElement(c); }); });
}
document.addEventListener('DOMContentLoaded', function(){
  var saved = null; try { saved = localStorage.getItem('boopDark'); } catch(e){}
  var dark = (saved === null) ? true : (saved === '1');
  var cb = document.getElementById('theme-toggle');
  if(cb) cb.checked = dark;
  boopApplyTheme(dark);
});
"""

daisy_hdrs = [
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'),
    *KatexMarkdownJS(),
    Link(id='hljs-theme', rel='stylesheet',
         href='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/styles/atom-one-dark.min.css'),
    Script(src='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/highlight.min.js'),
    Script(src='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/languages/python.min.js'),
    Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/codemirror@5/lib/codemirror.min.css'),
    Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/codemirror@5/theme/material-darker.min.css'),
    Script(src='https://cdn.jsdelivr.net/npm/codemirror@5/lib/codemirror.min.js'),
    Script(src='https://cdn.jsdelivr.net/npm/codemirror@5/mode/python/python.min.js'),
    Script(src='https://cdn.jsdelivr.net/npm/codemirror@5/addon/comment/comment.min.js'),
    Link(rel='icon', type='image/png', href='/logo.png'),
    Style(_MARKED_CSS),
    Script("import { AnsiUp } from 'https://cdn.jsdelivr.net/npm/ansi_up@6/ansi_up.js';"
           " window.AnsiUp = AnsiUp; if(window.boopRenderAnsi) boopRenderAnsi();", type='module'),
    Script(_THEME_JS),
    Script(_HOTKEYS_JS),
    Script(_EDIT_JS),
]

In [ ]:
#| export
app = FastHTML(hdrs=daisy_hdrs, htmlkw={'data-theme':'dark'})
rt  = app.route
p   = partial(HTMX, app=app, host=None, port=None)

## Code execution

A single IPython shell backs every code cell (like the lesson's `ex`), with errors
returned as text instead of raised.

In [ ]:
#| export
_shell = get_shell()
_shell.system = _shell.system_piped   # capture `!cmd` output into stdout

def run_code(src):
    "Execute `src` in the shared shell; return result, stdout, or an error string."
    res = _shell.run_cell(src)
    if res.error_in_exec is not None:
        e = res.error_in_exec
        return f'{type(e).__name__}: {e}'
    if res.result is not None: return res.result
    return (res.stdout or '').replace('\r\n', '\n')

## Cell + Notebook model

A `Cell` carries its type, source, optional output, and a `visible` flag (the eye
toggle — whether the LLM sees it). `Notebook` is the in-memory store of cells, the
composer's selected type, and the currently `selected` cell (for hotkeys).

In [ ]:
#| export
CTYPES = ('code','note','prompt','raw')  # types you can author; 'assistant' is generated

class Cell:
    def __init__(self, id, ctype, source, output=None, visible=True):
        self.id,self.ctype,self.source = id,ctype,source
        self.output,self.visible = output,visible
        self.ts = datetime.now().strftime('%I:%M:%S %p')

In [ ]:
#| export
class Notebook:
    def __init__(self):
        self.cells, self._nid, self.compose_type, self.selected = [], 0, 'code', None
        self.name = 'untitled'

    def insert_at(self, pos, ctype, source, output=None, visible=True):
        self._nid += 1
        c = Cell(self._nid, ctype, source, output, visible)
        self.cells.insert(pos, c)
        return c

    def add(self, ctype, source, output=None, visible=True):
        return self.insert_at(len(self.cells), ctype, source, output, visible)

    def index(self, id): return next((i for i,c in enumerate(self.cells) if c.id==id), None)
    def get(self, id):
        i = self.index(id)
        return self.cells[i] if i is not None else None
    def sel_index(self):
        return None if self.selected is None else self.index(self.selected)

    def remove(self, id):
        i = self.index(id)
        if i is not None: del self.cells[i]

    def move(self, id, delta):
        i = self.index(id)
        if i is None: return
        j = i + delta
        if 0 <= j < len(self.cells):
            self.cells[i], self.cells[j] = self.cells[j], self.cells[i]

nb = Notebook()

## The stubbed "LLM"

The whole point of the visibility toggle: context is only the *visible* cells. The
stub proves the plumbing by reporting what it can see; swap `stub_reply` for a real
model call later.

In [ ]:
#| export
def llm_context(nb):
    "Exactly what a real model would receive: the visible cells, in order."
    return '\n'.join(f'[{c.ctype}] {c.source}' for c in nb.cells if c.visible)

def stub_reply(nb, prompt):
    n = sum(c.visible for c in nb.cells)
    return (f'(stub) I can see {n} visible cell(s). You said: '
            f'"{prompt.strip()}". Wire a real model into stub_reply() later.')

## Rendering

Each type gets a colored left border (matching the SolveIt screenshot: raw=yellow,
code=blue, note=green, prompt/assistant=red). The selected cell gets a ring;
hidden-from-LLM cells are dimmed. Note cells render as markdown via the `.marked`
class (KaTeX, images, HTML).

In [ ]:
#| export
BORDER = {'raw':'border-warning', 'code':'border-info', 'note':'border-success',
          'prompt':'border-error', 'assistant':'border-error'}

def IconBtn(sym, title, **kw):
    return fh.Button(sym, cls='btn btn-sm btn-ghost', title=title, **kw)

In [ ]:
#| export
def cell_toolbar(c):
    tgt = '#notebook'
    copy_btn = fh.Button('📋', id=f'copy-{c.id}', title='Copy to clipboard', type='button',
                         cls='btn btn-sm btn-ghost', data_src=c.source,
                         onclick=f"boopCopy({c.id}, '{c.ctype}')")
    btns = [copy_btn, IconBtn('👁' if c.visible else '🚫',
                    'Hide from LLM' if c.visible else 'Show to LLM',
                    hx_post=toggle_vis.to(id=c.id), hx_target=tgt)]
    if c.ctype == 'code':
        btns.append(IconBtn('▶', 'Run', onclick=f'boopSave({c.id})'))
    elif c.ctype == 'prompt':
        btns.append(IconBtn('▶', 'Run', hx_post=run_cell.to(id=c.id), hx_target=tgt))
    btns += [
        IconBtn('↑', 'Move up',   hx_post=move_cell.to(id=c.id, delta=-1), hx_target=tgt),
        IconBtn('↓', 'Move down', hx_post=move_cell.to(id=c.id, delta=1),  hx_target=tgt),
        IconBtn('🗑', 'Delete', hx_post=del_cell.to(id=c.id), hx_target=tgt),
    ]
    return Div(*btns, cls='flex gap-1 ml-auto')

In [ ]:
#| export
def type_dropdown(c):
    "Click the cell-type word to switch it (code/note/prompt/raw). Scoped to just this cell."
    if c.ctype == 'assistant':
        return Span('Assistant', cls='font-semibold text-sm')
    opts = [Li(fh.A(t.capitalize(), hx_post=set_ctype.to(id=c.id, t=t),
                    hx_target=f'#cell-{c.id}', hx_swap='outerHTML'))
            for t in CTYPES if t != c.ctype]
    return Div(
        Div(c.ctype.capitalize(), tabindex='0', role='button',
            cls='font-semibold text-sm cursor-pointer'),
        Ul(*opts, tabindex='0', cls='dropdown-content menu bg-base-200 rounded-box z-10 w-28 p-1 shadow'),
        cls='dropdown dropdown-bottom')

def cell_header(c):
    rest = f': {c.id}' + (f' ({c.ts})' if c.ctype in ('code','assistant') else '')
    return Div(type_dropdown(c),
               Span(rest, cls='font-semibold text-sm cursor-pointer flex-1',
                    hx_post=select.to(id=c.id), hx_target='#notebook'),
               cell_toolbar(c), cls='flex items-center gap-2 mb-1')

def cell_body(c):
    "Note/raw/assistant bodies. Code cells never reach here -- render_cell() routes them to code_editor()."
    if c.ctype == 'note':                       # markdown + KaTeX + HTML + images
        return Div(c.source, cls='marked prose max-w-none')
    if c.ctype == 'raw':
        return Pre(c.source, cls='font-mono text-sm whitespace-pre-wrap')
    return Div(c.source, cls='text-sm whitespace-pre-wrap')

def _cell_outer(c, *content):
    dim    = '' if c.visible else 'opacity-40'
    indent = 'ml-8' if c.ctype == 'assistant' else ''
    ring   = 'ring-2 ring-primary ring-offset-2 ring-offset-base-100 rounded' if c.id == nb.selected else ''
    return Div(*content, id=f'cell-{c.id}',
               cls=f'border-l-4 {BORDER[c.ctype]} pl-3 py-2 my-2 {dim} {indent} {ring}')

def code_editor(c):
    "Code cells are always a live CodeMirror editor; Shift/Ctrl/Cmd+Enter or the play button runs."
    ta = Textarea(c.source, name='source', id=f'ta-{c.id}',
                  rows=str(max(2, c.source.count(chr(10)) + 1)),
                  cls='textarea textarea-bordered w-full font-mono',
                  data_cm='code', data_cid=str(c.id))
    parts = [Form(ta, hx_post=save_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML')]
    if c.output not in (None, ''):
        parts.append(Pre(str(c.output), cls='ansi-out text-sm mt-1 whitespace-pre overflow-x-auto'))
    return Div(*parts)

def render_cell(c):
    "Code cells are always editors; note/raw render and open an editor on click."
    if c.ctype == 'code':
        return _cell_outer(c, cell_header(c), code_editor(c))
    body = cell_body(c)
    if c.ctype != 'assistant':
        body = Div(body, cls='cursor-text',
                   hx_get=edit_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML')
    return _cell_outer(c, cell_header(c), body)

def render_cell_edit(c):
    "Inline editor. Code cells use CodeMirror (Python highlight, no wrap); notes/raw use a textarea. Shift/Ctrl/Cmd+Enter saves."
    ta = Textarea(c.source, name='source', id=f'ta-{c.id}',
                  rows=str(max(3, c.source.count(chr(10)) + 2)),
                  cls='textarea textarea-bordered w-full font-mono',
                  data_cm='edit', data_cid=str(c.id))
    buttons = Div(Button('Save', type='button', cls='btn btn-primary btn-xs',
                         onclick=f'boopSave({c.id})'),
                  Button('Cancel', type='button', cls='btn btn-ghost btn-xs',
                         hx_get=view_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML'),
                  cls='flex gap-2 justify-end mt-1')
    form = Form(ta, buttons, hx_post=save_cell.to(id=c.id),
                hx_target=f'#cell-{c.id}', hx_swap='outerHTML')
    return _cell_outer(c, cell_header(c), form)

def render_nb():
    return Div(*[render_cell(c) for c in nb.cells], id='notebook', cls='flex flex-col')

## Composer

The bottom bar: type tabs, a textarea, Submit. Picking a tab sets the type
server-side; Submit creates the cell (running it, if code; spawning an Assistant
reply, if prompt).

In [ ]:
#| export
def composer(draft='', oob=False):
    tabs = [fh.A(t.capitalize(),
                 cls=f'tab {"tab-active" if nb.compose_type==t else ""}',
                 hx_post=set_type.to(t=t), hx_target='#composer', hx_swap='outerHTML')
            for t in CTYPES]
    ta_kw = {'data_cm':'composer'} if nb.compose_type=='code' else {}
    div_kw = {'hx_swap_oob':'true'} if oob else {}
    return Div(
        Div(*tabs, cls='tabs tabs-boxed'),
        Form(Textarea(draft, placeholder=f'{nb.compose_type} cell…', name='source',
                      id='compose-input', rows='3',
                      onkeydown="if((event.shiftKey||event.ctrlKey||event.metaKey)&&event.key==='Enter')"
                               "{event.preventDefault();this.form.requestSubmit();}",
                      cls='textarea textarea-bordered w-full font-mono', **ta_kw),
             Div(Button('Submit', type='button', onclick='boopComposerSubmit()', cls='btn btn-primary'), cls='flex justify-end mt-2'),
             hx_post=submit_cell, hx_target='#notebook', hx_swap='beforeend'),
        id='composer', cls='border-t border-base-300 pt-3 mt-4', **div_kw)

def render_app(draft=''):
    return Div(render_nb(), composer(draft), id='app')

## Routes

Composer/toolbar routes plus the command-mode routes driven by hotkeys
(`select`, `select_delta`, `insert`, `del_selected`, `settype_selected`).

In [ ]:
#| export
# ---- top menu / control bar ----
def theme_swap():
    "DaisyUI sun/moon swap; drives boopApplyTheme (default dark)."
    return NotStr('<label class="swap swap-rotate btn btn-ghost btn-circle btn-sm" title="Toggle light/dark">'
      '<input type="checkbox" id="theme-toggle" onchange="boopThemeToggle(this)" checked />'
      '<svg class="swap-off h-5 w-5 fill-current" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
      '<path d="M5.64,17l-.71.71a1,1,0,0,0,0,1.41,1,1,0,0,0,1.41,0l.71-.71A1,1,0,0,0,5.64,17ZM5,12a1,1,0,0,0-1-1H3a1,1,0,0,0,0,2H4A1,1,0,0,0,5,12Zm7-7a1,1,0,0,0,1-1V3a1,1,0,0,0-2,0V4A1,1,0,0,0,12,5ZM5.64,7.05a1,1,0,0,0,.7.29,1,1,0,0,0,.71-.29,1,1,0,0,0,0-1.41l-.71-.71A1,1,0,0,0,4.93,6.34Zm12,.29a1,1,0,0,0,.7-.29l.71-.71a1,1,0,1,0-1.41-1.41L17,5.64a1,1,0,0,0,0,1.41A1,1,0,0,0,17.66,7.34ZM21,11H20a1,1,0,0,0,0,2h1a1,1,0,0,0,0-2Zm-9,8a1,1,0,0,0-1,1v1a1,1,0,0,0,2,0V20A1,1,0,0,0,12,19ZM18.36,17A1,1,0,0,0,17,18.36l.71.71a1,1,0,0,0,1.41,0,1,1,0,0,0,0-1.41ZM12,6.5A5.5,5.5,0,1,0,17.5,12,5.51,5.51,0,0,0,12,6.5Zm0,9A3.5,3.5,0,1,1,15.5,12,3.5,3.5,0,0,1,12,15.5Z"/></svg>'
      '<svg class="swap-on h-5 w-5 fill-current" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
      '<path d="M21.64,13a1,1,0,0,0-1.05-.14,8.05,8.05,0,0,1-3.37.73A8.15,8.15,0,0,1,9.08,5.49a8.59,8.59,0,0,1,.25-2A1,1,0,0,0,8,2.36,10.14,10.14,0,1,0,22,14.05,1,1,0,0,0,21.64,13Zm-9.5,6.69A8.14,8.14,0,0,1,7.08,5.22v.27A10.15,10.15,0,0,0,17.22,15.63a9.79,9.79,0,0,0,2.1-.22A8.11,8.11,0,0,1,12.14,19.73Z"/></svg></label>')

_BOOP2NB = {'code':'code', 'note':'markdown', 'prompt':'markdown', 'raw':'raw', 'assistant':'markdown'}

def save_notebook(path=None):
    "Serialize `nb.cells` to a real Jupyter notebook file (`{nb.name}.ipynb` in the cwd, by default)."
    path = Path(path) if path else Path.cwd()/f'{nb.name}.ipynb'
    doc = _nbf.v4.new_notebook()
    for c in nb.cells:
        meta = {'boopiter': {'ctype': c.ctype, 'visible': c.visible}}
        kind = _BOOP2NB.get(c.ctype, 'raw')
        if kind == 'code':
            outputs = [_nbf.v4.new_output('stream', name='stdout', text=str(c.output))] if c.output not in (None, '') else []
            cell = _nbf.v4.new_code_cell(c.source, outputs=outputs, metadata=meta)
        elif kind == 'markdown':
            cell = _nbf.v4.new_markdown_cell(c.source, metadata=meta)
        else:
            cell = _nbf.v4.new_raw_cell(c.source, metadata=meta)
        doc.cells.append(cell)
    _nbf.write(doc, str(path))
    return path

_NB_FALLBACK = {'code':'code', 'markdown':'note', 'raw':'raw'}

def load_notebook(path):
    "Load a Jupyter notebook file into `nb`, replacing its current contents. Inverse of save_notebook()."
    path = Path(path)
    doc = _nbf.read(str(path), as_version=4)
    nb.cells.clear()
    nb._nid = 0
    nb.selected = None
    for cell in doc.cells:
        meta = cell.get('metadata', {}).get('boopiter', {})
        ctype = meta.get('ctype')
        if ctype not in CTYPES + ('assistant',):
            ctype = _NB_FALLBACK.get(cell.cell_type, 'raw')  # plain (non-boopiter) notebook
        output = None
        if ctype == 'code':
            texts = [o.get('text','') for o in cell.get('outputs', []) if o.get('output_type') == 'stream']
            output = ''.join(texts) or None
        nb.add(ctype, cell.source, output=output, visible=meta.get('visible', True))
    nb.name = str(path.with_suffix(''))  # keep the directory, only strip .ipynb
    return nb

def fname_display():
    return Span(nb.name, id='fname', title='Click to rename',
                cls='cursor-pointer font-mono opacity-80 hover:opacity-100',
                hx_get=rename_form, hx_target='#fname', hx_swap='outerHTML')

@rt
def rename_form():
    return Form(Input(value=nb.name, name='name',
                      cls='input input-sm input-bordered font-mono',
                      onkeydown="if(event.key===\'Escape\'){this.form.requestSubmit();}"),
                Script("var i=document.querySelector(\'#fname input\'); if(i){i.focus();i.select();}"),
                id='fname', hx_post=rename, hx_target='#fname', hx_swap='outerHTML')

def top_bar():
    brand = Div(Img(src='/logo.png', cls='h-8 w-8 rounded-full'),
                Span('boopiter', cls='font-bold text-lg'),
                Span('/', cls='opacity-40'), fname_display(),
                cls='flex items-center gap-2')
    ctrls = Div(
        fh.Button('\u2297', title='Interrupt kernel', cls='btn btn-ghost btn-circle btn-sm text-base',
                  hx_post=interrupt_kernel, hx_swap='none'),
        fh.Button('\u21bb', title='Restart kernel', cls='btn btn-ghost btn-circle btn-sm text-base',
                  hx_post=restart_kernel, hx_swap='none'),
        theme_swap(), cls='flex items-center gap-1')
    return Div(brand, ctrls, cls='navbar bg-base-200 shadow px-4 flex justify-between shrink-0')

@rt('/_boopiter_ping')
def boopiter_ping():
    "Identity check so `boopiter launch` can tell a live boopiter instance apart from something else on the port."
    return 'boopiter'

@rt('/logo.png')
def logo_png():
    return FileResponse(Path(__file__).parent.parent/'images/logo.png')

@rt
def index():
    return (Title('boopiter'),
            Div(top_bar(),
                Div(Div(render_app(), cls='max-w-3xl mx-auto p-4'),
                    cls='flex-1 overflow-y-auto'),
                cls='h-screen flex flex-col'))

@rt
def save_now():
    save_notebook()
    return ''

@rt
def rename(name:str):
    "Rename and persist the notebook to `{new_name}.ipynb` in the server's cwd."
    nb.name = name.strip() or nb.name
    save_notebook()
    return fname_display()

@rt
def restart_kernel():
    _shell.reset()          # clear kernel namespace
    return ''

@rt
def interrupt_kernel():
    return ''               # placeholder: true interrupt needs threaded/async execution

@rt
def set_type(t:str):
    if t in CTYPES: nb.compose_type = t
    return composer()

def add_cell(t, source):
    "Create a cell of type `t`: run it if code, spawn an assistant reply if prompt. Returns the new cell(s)."
    if t == 'code':
        return [nb.add('code', source, output=run_code(source))]
    elif t == 'prompt':
        c1 = nb.add('prompt', source)
        c2 = nb.add('assistant', stub_reply(nb, source))
        return [c1, c2]
    else:
        return [nb.add(t, source)]

@rt
def submit_cell(source:str):
    "Append only the new cell(s) to #notebook and reset the composer out-of-band, so untouched cells' editors are never re-created."
    new = add_cell(nb.compose_type, source) if source.strip() else []
    return *[render_cell(c) for c in new], composer(oob=True)

@rt
def split(source:str, pos:int):
    "Split the composer at the caret: head becomes a cell, tail stays in the composer."
    head, tail = source[:pos], source[pos:]
    new = add_cell(nb.compose_type, head) if head.strip() else []
    return *[render_cell(c) for c in new], composer(draft=tail, oob=True)

@rt
def run_cell(id:int):
    c = nb.get(id)
    if c and c.ctype == 'code': c.output = run_code(c.source)
    return render_nb()

@rt
def toggle_vis(id:int):
    c = nb.get(id)
    if c: c.visible = not c.visible
    return render_nb()

@rt
def del_cell(id:int):
    nb.remove(id)
    return render_nb()

@rt
def move_cell(id:int, delta:int):
    nb.move(id, delta)
    return render_nb()

In [ ]:
#| export
# --- command-mode (hotkey) routes ---
@rt
def select(id:int):
    nb.selected = id
    return render_nb()

@rt
def select_delta(delta:int):
    if nb.cells:
        i = nb.sel_index()
        i = (0 if delta > 0 else len(nb.cells)-1) if i is None else min(max(i+delta, 0), len(nb.cells)-1)
        nb.selected = nb.cells[i].id
    return render_nb()

@rt
def insert(where:str):
    i = nb.sel_index()
    pos = len(nb.cells) if i is None else (i if where == 'above' else i+1)
    nb.selected = nb.insert_at(pos, nb.compose_type, '').id
    return render_nb()

@rt
def del_selected():
    i = nb.sel_index()
    if i is not None:
        del nb.cells[i]
        nb.selected = nb.cells[min(i, len(nb.cells)-1)].id if nb.cells else None
    return render_nb()

@rt
def settype_selected(t:str):
    c = nb.get(nb.selected) if nb.selected is not None else None
    if c and t in CTYPES: c.ctype = t
    return render_nb()

@rt
def set_ctype(id:int, t:str):
    "Change one cell's type in place; returns just that cell so the rest of the notebook is untouched."
    c = nb.get(id)
    if c and t in CTYPES and t != c.ctype:
        c.ctype = t
        c.output = None  # stale output no longer meaningful under the new type
    return render_cell(c) if c else render_nb()

# --- inline editing ---
@rt
def edit_cell(id:int):
    c = nb.get(id)
    if not c: return render_nb()
    if c.ctype == 'assistant': return render_cell(c)
    nb.selected = id
    return render_cell_edit(c)

@rt
def view_cell(id:int):
    c = nb.get(id)
    return render_cell(c) if c else render_nb()

@rt
def save_cell(id:int, source:str):
    c = nb.get(id)
    if c:
        c.source = source
        if c.ctype == 'code': c.output = run_code(source)
    return render_cell(c) if c else render_nb()

## Run it

In a notebook, start the server and preview inline. Click a cell's header to
select it, then use command-mode keys: `A`/`B` insert above/below, `D D` delete,
`J`/`K` (or arrows) move selection, `M`/`Y`/`R` change type. In the composer,
`Cmd/Ctrl+/` toggles comments on the selected lines and `Cmd/Ctrl+Shift+-` splits
at the caret. On WSL see the lesson's port notes for reaching it from Windows.

In [ ]:
srv = JupyUvi(app)
p(index())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()